In [12]:
import glob as glob
import h5py
from datetime import datetime, timedelta,timezone
import numpy as np
import os

## Load data, apply masks and find reference channel

In [13]:
# ============================================================
# PARAMETERS
# ============================================================

# --- Paths ---
PATHS_FAROE = {
    'das_folder': '/Users/emil/Documents/GitHub/DAS/Data/20240512',
    'cable_json': '/Users/emil/Documents/DAS-ft-data/cable.json',
    'ais_csv':    '/Users/emil/Documents/GitHub/DAS/Data/SHEFA_20240512092543_unknown_5.csv',
}

# --- Ship ---
ship_name       = 'KAPITAN NAZIN'
mmsi            = 273332560
crossing_time   = '092610'
heading         = 'south'

# --- Files ---
file_list_faroe = sorted(glob.glob(f"{PATHS_FAROE['das_folder']}/*.hdf5"))

# --- Time window ---
year        = 2024
month       = 5
day         = 12
time_start  = '092410'
time_end    = '092810'

# --- Array ---
ref_distance_km = 18.45 #km
array_min_km    = ref_distance_km - 0.5
array_max_km    = ref_distance_km + 0.5
time_axis_downsampling_step = 1

# --- Grid ---
grid_center_lat  = 61.97126
grid_center_lon  = -6.52650
grid_spacing_m   = 10
grid_extent_ns_m = 1500
grid_extent_we_m = 750

# --- MUSIC ---
fc      = 48.0   # Hz
bw      = 2.0    # Hz
c_water = 1500   # m/s

window_s    = 4.0  # window length in seconds
step_s      = 1.0  # step between windows in seconds
threshold   = 0.999 # minimum peak value to accept

# --- Cable ---
offset_km = -0.1

In [14]:
import h5py
import glob
import os
import numpy as np
from datetime import datetime, timedelta, timezone

# ============================================================
# PEEK FIRST FILE — get metadata
# ============================================================
with h5py.File(file_list_faroe[0], 'r') as f:
    dx       = float(f['header/dx'][()])
    dt       = float(f['header/dt'][()])
    fs       = 1 / dt
    channels = f['header/channels'][()]
    n_samples = f['data'].shape[0]

n_channels            = len(channels)
raw_distance_array_km = channels * dx / 1000
spatial_mask          = (raw_distance_array_km >= array_min_km) & (raw_distance_array_km <= array_max_km)
distance_array_km     = raw_distance_array_km[spatial_mask]

print(f"dx:                    {dx:.4f} m")
print(f"fs:                    {fs:.1f} Hz")
print(f"Total channels:        {n_channels}")
print(f"Array window:          {array_min_km} – {array_max_km} km")
print(f"Channels in window:    {spatial_mask.sum()}")
print(f"Channel distance range:{distance_array_km[0]:.4f} – {distance_array_km[-1]:.4f} km")

# ============================================================
# TIME AXIS + DATA LOADING
# ============================================================
def hhmmss_to_dt(hhmmss_str, year, month, day):
    return datetime(year, month, day,
                    int(hhmmss_str[:2]), int(hhmmss_str[2:4]), int(hhmmss_str[4:6]),
                    tzinfo=timezone.utc)

time_start_datetime = hhmmss_to_dt(time_start, year, month, day)
time_end_datetime   = hhmmss_to_dt(time_end,   year, month, day)

print(f"\nRequested time window: {time_start_datetime} – {time_end_datetime}")
print(f"Samples per file:      {n_samples}  ({n_samples / fs:.1f} s)")

all_datetimes = []
data_list     = []
files_loaded  = []

for file in file_list_faroe:
    filestart_hhmmss_str = os.path.basename(file).replace('.hdf5', '')
    filestart_datetime   = hhmmss_to_dt(filestart_hhmmss_str, year, month, day)

    file_ends_before_window  = filestart_datetime + timedelta(seconds=10) < time_start_datetime
    file_starts_after_window = filestart_datetime > time_end_datetime

    if not file_ends_before_window and not file_starts_after_window:
        with h5py.File(file, 'r') as f:
            file_start_time  = float(f['header/time'][()])
            file_start_dt    = datetime.fromtimestamp(file_start_time, tz=timezone.utc)
            file_n_samples   = f['data'].shape[0]
            data_chunk       = f['data'][::time_axis_downsampling_step, spatial_mask]

        filetimes = [file_start_dt + timedelta(seconds=i * dt)
                     for i in range(0, file_n_samples, time_axis_downsampling_step)]
        all_datetimes.extend(filetimes)
        data_list.append(data_chunk)
        files_loaded.append(os.path.basename(file))

print(f"\nFiles loaded ({len(files_loaded)}):")
for f in files_loaded:
    print(f"  {f}")

all_datetimes_arr = np.array(all_datetimes)
data_raw          = np.vstack(data_list)
time_mask         = (all_datetimes_arr >= time_start_datetime) & (all_datetimes_arr <= time_end_datetime)
time_axis         = all_datetimes_arr[time_mask]
data              = data_raw[time_mask]

print(f"\nAfter time mask:")
print(f"  data shape:            {data.shape}  (samples × channels)")
print(f"  time_axis shape:       {time_axis.shape}")
print(f"  First sample:          {time_axis[0]}")
print(f"  Last sample:           {time_axis[-1]}")
print(f"  Actual duration:       {(time_axis[-1] - time_axis[0]).total_seconds():.2f} s")

# Reference channel
reference_search_raw         = np.abs(raw_distance_array_km - ref_distance_km)
reference_channel_idx_raw    = np.argmin(reference_search_raw)
reference_channel_idx_masked = np.argmin(reference_search_raw[spatial_mask])

dx:                    1.0213 m
fs:                    800.0 Hz
Total channels:        15624
Array window:          17.95 – 18.95 km
Channels in window:    245
Channel distance range:17.9504 – 18.9472 km

Requested time window: 2024-05-12 09:24:10+00:00 – 2024-05-12 09:28:10+00:00
Samples per file:      8000  (10.0 s)

Files loaded (25):
  092405.hdf5
  092415.hdf5
  092425.hdf5
  092435.hdf5
  092445.hdf5
  092455.hdf5
  092505.hdf5
  092515.hdf5
  092525.hdf5
  092535.hdf5
  092545.hdf5
  092555.hdf5
  092605.hdf5
  092615.hdf5
  092625.hdf5
  092635.hdf5
  092645.hdf5
  092655.hdf5
  092705.hdf5
  092715.hdf5
  092725.hdf5
  092735.hdf5
  092745.hdf5
  092755.hdf5
  092805.hdf5

After time mask:
  data shape:            (192001, 245)  (samples × channels)
  time_axis shape:       (192001,)
  First sample:          2024-05-12 09:24:10.000750+00:00
  Last sample:           2024-05-12 09:28:09.999750+00:00
  Actual duration:       240.00 s


# Load cable

In [15]:
import json
import pandas as pd

cable_path = PATHS_FAROE['cable_json']
with open(cable_path, 'r') as f:
    cable_data = json.load(f)
cable_positions_dict = cable_data.get('positions', cable_data)

cable_lat_full   = []
cable_lon_full   = []
cable_depth_full = []

for i in range(len(cable_positions_dict)):
    value = cable_positions_dict[str(i + 1)]
    cable_lat_full.append(value['lat'])
    cable_lon_full.append(value['lon'])
    cable_depth_full.append(value['depth'])

cable_lat_full   = np.array(cable_lat_full)
cable_lon_full   = np.array(cable_lon_full)
cable_depth_full = np.array(cable_depth_full)

cable_df = pd.DataFrame({'lon': cable_lon_full, 'lat': cable_lat_full, 'depth': cable_depth_full})

offset_channels = int(round(offset_km * 1000 / dx))
masked_indices  = np.where(spatial_mask)[0]
offset_indices  = masked_indices + offset_channels

cable_lat   = cable_lat_full[offset_indices]
cable_lon   = cable_lon_full[offset_indices]
cable_depth = np.abs(cable_depth_full[offset_indices])

print(f"Offset:          {offset_km} km = {offset_channels} channels")
print(f"cable_lat shape: {cable_lat.shape}")
print(f"\nFirst channel: lat={cable_lat[0]:.5f}, lon={cable_lon[0]:.5f}, depth={cable_depth[0]:.1f} m")
print(f"Last channel:  lat={cable_lat[-1]:.5f}, lon={cable_lon[-1]:.5f}, depth={cable_depth[-1]:.1f} m")
print(f"Ref channel:   lat={cable_lat[reference_channel_idx_masked]:.5f}, "
      f"lon={cable_lon[reference_channel_idx_masked]:.5f}, "
      f"depth={cable_depth[reference_channel_idx_masked]:.1f} m")

Offset:          -0.1 km = -98 channels
cable_lat shape: (245,)

First channel: lat=61.97451, lon=-6.53304, depth=58.0 m
Last channel:  lat=61.96802, lon=-6.51997, depth=53.0 m
Ref channel:   lat=61.97126, lon=-6.52650, depth=55.5 m


## Load AIS data

In [16]:
ais_csv_path = PATHS_FAROE['ais_csv']

ais_ship = pd.read_csv(ais_csv_path)
ais_ship['timestamp'] = pd.to_datetime(ais_ship['timestamp'], format='ISO8601')

ship_timestamps      = ais_ship['timestamp'].values
ship_lats            = ais_ship['lat'].values
ship_lons            = ais_ship['lon'].values
ship_dist_from_cable = ais_ship['dist'].values
ship_sog             = ais_ship['sog'].values

print(f"AIS pings:          {len(ship_lats)}")
print(f"Time range:         {ais_ship['timestamp'].iloc[0]} – {ais_ship['timestamp'].iloc[-1]}")
print(f"Lat range:          {ship_lats.min():.5f} – {ship_lats.max():.5f}")
print(f"Lon range:          {ship_lons.min():.5f} – {ship_lons.max():.5f}")
print(f"Min dist to cable:  {ship_dist_from_cable.min():.1f} m")
print(f"Max dist to cable:  {ship_dist_from_cable.max():.1f} m")

apex_idx = np.argmin(ship_dist_from_cable)
print(f"\nApex (closest point to cable):")
print(f"  Time:     {ais_ship['timestamp'].iloc[apex_idx]}")
print(f"  Distance: {ship_dist_from_cable[apex_idx]:.1f} m")
print(f"  Lat/Lon:  {ship_lats[apex_idx]:.5f}, {ship_lons[apex_idx]:.5f}")
print(f"  SOG:      {ship_sog[apex_idx]:.1f} kn")

AIS pings:          181
Time range:         2024-05-12 09:23:00 – 2024-05-12 09:29:00
Lat range:          61.96138 – 61.98235
Lon range:          -6.52936 – -6.52326
Min dist to cable:  1.7 m
Max dist to cable:  970.7 m

Apex (closest point to cable):
  Time:     2024-05-12 09:26:10
  Distance: 1.7 m
  Lat/Lon:  61.97130, -6.52655
  SOG:      12.6 kn


## MUSIC

#### Grid

In [17]:
def to_utm(lon, lat, ref_lat=grid_center_lat, ref_lon=grid_center_lon):
    R = 6371000
    x = R * np.radians(np.array(lon) - ref_lon) * np.cos(np.radians(ref_lat))
    y = R * np.radians(np.array(lat) - ref_lat)
    return x, y

def from_utm(x, y, ref_lat=grid_center_lat, ref_lon=grid_center_lon):
    R = 6371000
    lat = ref_lat + np.degrees(np.array(y) / R)
    lon = ref_lon + np.degrees(np.array(x) / (R * np.cos(np.radians(ref_lat))))
    return lon, lat

# --- Grid rotation ---
grid_rotation_deg = 7 # clockwise degrees

# Project grid centre to local UTM
centre_x, centre_y = to_utm(grid_center_lon, grid_center_lat)

# Build 1-D UTM axes
x_grid = np.arange(centre_x - grid_extent_we_m,
                   centre_x + grid_extent_we_m,
                   grid_spacing_m)
y_grid = np.arange(centre_y - grid_extent_ns_m,
                   centre_y + grid_extent_ns_m,
                   grid_spacing_m)

# 2-D meshgrid in UTM
xx_utm, yy_utm = np.meshgrid(x_grid, y_grid)

# --- Rotate grid points around centre ---
angle_rad = np.radians(-grid_rotation_deg)  # negative = clockwise
dx_grid   = xx_utm - centre_x
dy_grid   = yy_utm - centre_y

xx_utm_rot = centre_x + dx_grid * np.cos(angle_rad) - dy_grid * np.sin(angle_rad)
yy_utm_rot = centre_y + dx_grid * np.sin(angle_rad) + dy_grid * np.cos(angle_rad)

# Back-project to lat/lon
lon_grid_2d, lat_grid_2d = from_utm(xx_utm_rot, yy_utm_rot)

# 1-D lat/lon axes
lat_grid = lat_grid_2d[:, 0]
lon_grid = lon_grid_2d[0, :]

# Flattened grid points
grid_points_utm = np.stack([xx_utm_rot.ravel(),
                             yy_utm_rot.ravel(),
                             np.zeros(xx_utm_rot.size)], axis=1)
grid_points_ll  = np.stack([lat_grid_2d.ravel(),
                             lon_grid_2d.ravel(),
                             np.zeros(xx_utm_rot.size)], axis=1)

# North/south mask based on cable orientation
cable_x, cable_y = to_utm(cable_lon, cable_lat)
m_cable, b_cable = np.polyfit(cable_x, cable_y, deg=1)
signed_dist = grid_points_utm[:, 1] - (m_cable * grid_points_utm[:, 0] + b_cable)
north_mask  = signed_dist > 0
south_mask  = signed_dist < 0

print(f"Grid: {len(y_grid)}×{len(x_grid)} = {xx_utm_rot.size} points")
print(f"Grid rotation: {grid_rotation_deg}° clockwise")
print(f"North mask: {north_mask.sum()} points")
print(f"South mask: {south_mask.sum()} points")

Grid: 300×150 = 45000 points
Grid rotation: 7° clockwise
North mask: 22360 points
South mask: 22640 points


In [18]:
print(f"Ref channel: lat={cable_lat[reference_channel_idx_masked]:.5f}, "
      f"lon={cable_lon[reference_channel_idx_masked]:.5f}")

Ref channel: lat=61.97126, lon=-6.52650


#### Steering delays

In [19]:
print(f"cable_lon.shape: {cable_lon.shape}")
print(f"cable_lat.shape: {cable_lat.shape}")
print(f"cable_depth.shape: {cable_depth.shape}")
print(f"data.shape: {data.shape}")

cable_lon.shape: (245,)
cable_lat.shape: (245,)
cable_depth.shape: (245,)
data.shape: (192001, 245)


In [20]:
# Convert cable positions to local UTM
cable_x, cable_y = to_utm(cable_lon, cable_lat)
ref_x, ref_y     = cable_x[reference_channel_idx_masked], cable_y[reference_channel_idx_masked]
ref_z            = cable_depth[reference_channel_idx_masked]

# Grid points in UTM (surface, z=0)
gx = grid_points_utm[:, 0]  # shape (n_points,)
gy = grid_points_utm[:, 1]  # shape (n_points,)

# Travel times from each grid point to each channel
# delta_x, delta_y, delta_z: shape (n_points, n_channels)
delta_x = gx[:, np.newaxis] - cable_x[np.newaxis, :]
delta_y = gy[:, np.newaxis] - cable_y[np.newaxis, :]
delta_z = cable_depth[np.newaxis, :]  # ship at surface, cable at depth

travel_times    = np.sqrt(delta_x**2 + delta_y**2 + delta_z**2) / c_water
steering_delays = travel_times - travel_times[:, reference_channel_idx_masked:reference_channel_idx_masked+1]

print(f"steering_delays shape: {steering_delays.shape}")
print(f"steering_delays min:   {steering_delays.min()*1000:.2f} ms")
print(f"steering_delays max:   {steering_delays.max()*1000:.2f} ms")
print(f"Ref channel delay:     {steering_delays[:, reference_channel_idx_masked].mean()*1000:.4f} ms (should be 0)")

steering_delays shape: (45000, 245)
steering_delays min:   -330.38 ms
steering_delays max:   331.03 ms
Ref channel delay:     0.0000 ms (should be 0)


#### MUSIC loop

In [21]:
from scipy.signal import butter, sosfiltfilt
from pyproj import Geod
geod = Geod(ellps='WGS84')

def bandpass(data, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    if highcut >= nyquist:
        highcut = nyquist * 0.99
    sos = butter(order, [lowcut, highcut], btype='bandpass', output='sos', fs=fs)
    return sosfiltfilt(sos, data, axis=0)

In [22]:
from arlpy import bf
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import json

# ============================================================
# OUTPUT DIRECTORY
# ============================================================
ship_tag    = ship_name.replace(' ', '_')
output_dir  = f'/Users/emil/Desktop/results'
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# PROJECT TO CABLE X — along-cable distance from cable start
# ============================================================
def project_to_cable_x(lon, lat, cable_lons, cable_lats):
    """Project a single point onto the cable polyline.
    Returns along-cable distance in metres from the first cable node."""
    lon0, lat0 = cable_lons[0], cable_lats[0]
    n = len(cable_lons)
    faz, _, d = geod.inv(
        np.full(n, lon0), np.full(n, lat0),
        cable_lons, cable_lats)
    cx = d * np.sin(np.radians(faz))
    cy = d * np.cos(np.radians(faz))
    cable_dist = np.concatenate([[0],
                  np.cumsum(np.sqrt(np.diff(cx)**2 + np.diff(cy)**2))])
    faz_p, _, d_p = geod.inv(lon0, lat0, lon, lat)
    px = d_p * np.sin(np.radians(faz_p))
    py = d_p * np.cos(np.radians(faz_p))
    dx_s   = np.diff(cx)
    dy_s   = np.diff(cy)
    seg_sq = np.where(dx_s**2 + dy_s**2 < 1e-10, np.inf, dx_s**2 + dy_s**2)
    t      = np.clip(((px - cx[:-1]) * dx_s + (py - cy[:-1]) * dy_s) / seg_sq,
                     0.0, 1.0)
    fx     = cx[:-1] + t * dx_s
    fy     = cy[:-1] + t * dy_s
    best_i = np.argmin((px - fx)**2 + (py - fy)**2)
    return float(cable_dist[best_i] +
                 t[best_i] * (cable_dist[best_i+1] - cable_dist[best_i]))

# ============================================================
# AIS INTERPOLATORS (lat, lon, dist)
# ============================================================
ais_timestamps_s  = np.array([pd.Timestamp(t).timestamp() for t in ship_timestamps])
ais_dist_interp   = interp1d(ais_timestamps_s, ship_dist_from_cable,
                              bounds_error=False, fill_value='extrapolate')
ais_lat_interp    = interp1d(ais_timestamps_s, ship_lats,
                              bounds_error=False, fill_value='extrapolate')
ais_lon_interp    = interp1d(ais_timestamps_s, ship_lons,
                              bounds_error=False, fill_value='extrapolate')

# AIS along-cable interpolator — computed once
ais_cable_x_arr = np.array([
    project_to_cable_x(lon, lat, cable_lon_full, cable_lat_full)
    for lon, lat in zip(ship_lons, ship_lats)
])
ais_cable_x_interp = interp1d(ais_timestamps_s, ais_cable_x_arr,
                               bounds_error=False, fill_value='extrapolate')

# ============================================================
# WINDOW SETUP
# ============================================================
window_samples = int(window_s * fs)
step_samples   = int(step_s   * fs)
n_windows      = (len(time_axis) - window_samples) // step_samples + 1

crossing_dt    = datetime(year, month, day,
                           int(crossing_time[:2]),
                           int(crossing_time[2:4]),
                           int(crossing_time[4:6]),
                           tzinfo=timezone.utc)
crossing_dt_np = np.datetime64(crossing_dt.replace(tzinfo=None))

print(f"Window length:  {window_s} s = {window_samples} samples")
print(f"Step:           {step_s} s = {step_samples} samples")
print(f"Total windows:  {n_windows}")
print(f"Crossing time:  {crossing_dt}")
print(f"Output dir:     {output_dir}")

# ============================================================
# MAIN LOOP
# ============================================================
BAR_W  = 28
idx_w  = len(str(n_windows))

results    = []
n_accepted = 0

for win_idx in range(n_windows):
    t_start = win_idx * step_samples
    t_end   = t_start + window_samples

    data_win          = data[t_start:t_end, :]
    data_bp_win       = bandpass(data_win, fc - bw/2, fc + bw/2, fs)
    data_analytic_win = hilbert(data_bp_win, axis=0).T

    if win_idx == 0:
        print(f"\nFirst window:")
        print(f"  data_win.shape:          {data_win.shape}")
        print(f"  data_analytic_win.shape: {data_analytic_win.shape}")
        print(f"  steering_delays.shape:   {steering_delays.shape}")

    window_time = time_axis[t_start]
    window_ts   = pd.Timestamp(window_time).timestamp()

    if heading == 'north':
        side_mask = north_mask if window_time >= crossing_dt else south_mask
    else:
        side_mask = south_mask if window_time >= crossing_dt else north_mask

    spectrum = bf.music(data_analytic_win, fc, steering_delays, nsignals=1)

    s_min, s_max = spectrum.min(), spectrum.max()
    if s_max <= s_min:
        results.append({
            'window_idx':    win_idx,
            'time':          window_time,
            'music_lat':     np.nan, 'music_lon': np.nan,
            'music_dist':    np.nan, 'music_cable_x': np.nan,
            'ais_lat':       float(ais_lat_interp(window_ts)),
            'ais_lon':       float(ais_lon_interp(window_ts)),
            'ais_dist':      float(ais_dist_interp(window_ts)),
            'ais_cable_x':   float(ais_cable_x_interp(window_ts)),
            'error_y':       np.nan, 'error_x': np.nan,
            'error_euclidean': np.nan,
            'peak_val':      0.0, 'accepted': False,
        })
        filled = int(BAR_W * (win_idx + 1) / n_windows)
        print(f'\r  [{"█"*filled}{"░"*(BAR_W-filled)}]  {win_idx+1:{idx_w}}/{n_windows}  rejected        (✓ {n_accepted}/{win_idx+1})',
              end='', flush=True)
        continue

    spectrum_norm = (spectrum - s_min) / (s_max - s_min)
    side_spectrum = spectrum_norm[side_mask]
    peak_val      = side_spectrum.max()

    # AIS position at this window
    ais_lat_now    = float(ais_lat_interp(window_ts))
    ais_lon_now    = float(ais_lon_interp(window_ts))
    ais_dist_now   = float(ais_dist_interp(window_ts))
    ais_cable_x_now = float(ais_cable_x_interp(window_ts))

    if peak_val >= threshold:
        side_indices = np.where(side_mask)[0]
        peak_idx     = side_indices[np.argmax(side_spectrum)]
        music_lat    = grid_points_ll[peak_idx, 0]
        music_lon    = grid_points_ll[peak_idx, 1]

        # RMSE_y: cross-cable distance error
        _, _, dists_to_cable = geod.inv(
            np.full(len(cable_lon_full), music_lon),
            np.full(len(cable_lat_full), music_lat),
            cable_lon_full, cable_lat_full)
        music_dist   = float(np.array(dists_to_cable).min())
        error_y      = abs(music_dist - ais_dist_now)

        # RMSE_x: along-cable error via cable projection
        music_cable_x = project_to_cable_x(music_lon, music_lat,
                                            cable_lon_full, cable_lat_full)
        error_x       = abs(music_cable_x - ais_cable_x_now)

        # Euclidean
        music_x, music_y = to_utm(music_lon, music_lat)
        ais_x,   ais_y   = to_utm(ais_lon_now, ais_lat_now)
        error_euclidean  = float(np.sqrt((music_x - ais_x)**2 +
                                          (music_y - ais_y)**2))
        n_accepted += 1
        accepted = True
    else:
        music_lat = music_lon = music_dist = music_cable_x = np.nan
        error_y = error_x = error_euclidean = np.nan
        accepted = False

    results.append({
        'window_idx':     win_idx,
        'time':           window_time,
        'music_lat':      music_lat,
        'music_lon':      music_lon,
        'music_dist':     music_dist,
        'music_cable_x':  music_cable_x,
        'ais_lat':        ais_lat_now,
        'ais_lon':        ais_lon_now,
        'ais_dist':       ais_dist_now,
        'ais_cable_x':    ais_cable_x_now,
        'error_y':        error_y,
        'error_x':        error_x,
        'error_euclidean': error_euclidean,
        'peak_val':       peak_val,
        'accepted':       accepted,
    })

    filled = int(BAR_W * (win_idx + 1) / n_windows)
    if accepted:
        info = f'euc={error_euclidean:.0f} m  (✓ {n_accepted}/{win_idx+1})'
    else:
        info = f'rejected        (✓ {n_accepted}/{win_idx+1})'
    print(f'\r  [{"█"*filled}{"░"*(BAR_W-filled)}]  {win_idx+1:{idx_w}}/{n_windows}  {info}',
          end='', flush=True)

print(f'\r  [{"█"*BAR_W}]  {n_windows}/{n_windows}  ✓ {n_accepted}/{n_windows} accepted                    ')

# ============================================================
# SUMMARY STATISTICS
# ============================================================
df = pd.DataFrame(results)
df['time'] = df['time'].astype(str)

accepted_df       = df[df['accepted']].reset_index(drop=True)
accepted_df_times = pd.to_datetime(accepted_df['time'], utc=True)
crossing_dt_pd    = pd.Timestamp(crossing_dt)
before_df         = accepted_df[accepted_df_times < crossing_dt_pd]
after_df          = accepted_df[accepted_df_times >= crossing_dt_pd]

def compute_stats(arr):
    arr = np.array(arr.dropna())
    if len(arr) == 0:
        return dict(rmse=np.nan, median=np.nan, std=np.nan, n=0)
    return dict(rmse=round(float(np.sqrt(np.mean(arr**2))), 1),
                median=round(float(np.median(arr)), 1),
                std=round(float(np.std(arr)), 1),
                n=len(arr))

summary = {
    'ship_name':  ship_name,
    'fc':         fc,
    'n_windows':  n_windows,
    'n_accepted': n_accepted,
    'all': {
        'error_y':         compute_stats(accepted_df['error_y']),
        'error_x':         compute_stats(accepted_df['error_x']),
        'error_euclidean': compute_stats(accepted_df['error_euclidean']),
    },
    'before': {
        'error_y':         compute_stats(before_df['error_y']),
        'error_x':         compute_stats(before_df['error_x']),
        'error_euclidean': compute_stats(before_df['error_euclidean']),
    },
    'after': {
        'error_y':         compute_stats(after_df['error_y']),
        'error_x':         compute_stats(after_df['error_x']),
        'error_euclidean': compute_stats(after_df['error_euclidean']),
    },
}

# ============================================================
# SAVE RESULTS
# ============================================================
df.to_csv(os.path.join(output_dir, 'results.csv'), index=False)
with open(os.path.join(output_dir, 'summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f"\nSaved to {output_dir}")
print(f"\n--- SUMMARY ({ship_name} | fc={fc} Hz) ---")
for phase in ['all', 'before', 'after']:
    s = summary[phase]
    print(f"\n  {phase.upper()}:")
    print(f"    Euclidean — RMSE: {s['error_euclidean']['rmse']} m  "
          f"Median: {s['error_euclidean']['median']} m  "
          f"Std: {s['error_euclidean']['std']} m  (n={s['error_euclidean']['n']})")
    print(f"    Cross-cable (y) — RMSE: {s['error_y']['rmse']} m  "
          f"Median: {s['error_y']['median']} m  "
          f"Std: {s['error_y']['std']} m")
    print(f"    Along-cable (x) — RMSE: {s['error_x']['rmse']} m  "
          f"Median: {s['error_x']['median']} m  "
          f"Std: {s['error_x']['std']} m")

# ============================================================
# FLAT SUMMARY CSV
# ============================================================
flat = {
    'ship_name':  ship_name,
    'fc':         fc,
    'n_windows':  n_windows,
    'n_accepted': n_accepted,
}
for phase in ['all', 'before', 'after']:
    for metric in ['error_y', 'error_x', 'error_euclidean']:
        for stat in ['rmse', 'median', 'std']:
            flat[f'{metric}_{stat}_{phase}'] = summary[phase][metric][stat]

flat_df = pd.DataFrame([flat])
flat_df.to_csv(os.path.join(output_dir, 'summary_flat.csv'), index=False)
print(f"Flat summary saved to {output_dir}/summary_flat.csv")

Window length:  4.0 s = 3200 samples
Step:           1.0 s = 800 samples
Total windows:  237
Crossing time:  2024-05-12 09:26:10+00:00
Output dir:     /Users/emil/Desktop/results

First window:
  data_win.shape:          (3200, 245)
  data_analytic_win.shape: (245, 3200)
  steering_delays.shape:   (45000, 245)
  [████████████████████████████]  237/237  ✓ 146/237 accepted                    

Saved to /Users/emil/Desktop/results

--- SUMMARY (KAPITAN NAZIN | fc=48.0 Hz) ---

  ALL:
    Euclidean — RMSE: 462.7 m  Median: 196.6 m  Std: 333.1 m  (n=146)
    Cross-cable (y) — RMSE: 360.1 m  Median: 149.3 m  Std: 269.1 m
    Along-cable (x) — RMSE: 291.0 m  Median: 99.3 m  Std: 220.7 m

  BEFORE:
    Euclidean — RMSE: 412.7 m  Median: 237.2 m  Std: 256.5 m  (n=80)
    Cross-cable (y) — RMSE: 295.2 m  Median: 164.6 m  Std: 180.9 m
    Along-cable (x) — RMSE: 290.0 m  Median: 99.3 m  Std: 214.2 m

  AFTER:
    Euclidean — RMSE: 516.9 m  Median: 160.9 m  Std: 407.2 m  (n=66)
    Cross-cable (y)